# Retail Sales Analytics with DuckDB

## Project Overview

This project analyzes retail sales transactions from a national superstore using DuckDB, SQL, and Python. The objective is to identify key drivers of revenue and profitability across product categories, customer segments, and geographic regions.

The analysis explores questions such as:

* Which product categories generate the most revenue?
* Which categories generate the most profit?
* How do discounts impact profitability?
* Which customer segments contribute the most value?
* Which regions and states perform best?

The project demonstrates data cleaning, SQL analytics, exploratory data analysis, and business-focused reporting using modern Python data tools.


## Data Source

This project uses the Superstore Dataset published on Kaggle by Vivek Chowdhury.

Dataset:
https://www.kaggle.com/datasets/vivek468/superstore-dataset-final

The dataset contains retail sales transactions including customer information, product categories, sales, discounts, and profit metrics. The data is commonly used for business intelligence, data visualization, and analytics projects.

The original dataset was downloaded from Kaggle and stored as a raw source file within the project's data directory.


## Project Breakdown

The architecture of this project is sectioned into three levels. These three stages include: Data Preprocessing using Python and Pandas, Feature Engineering using DuckDB SQL, and Business Analytics using DuckDB SQL. 

In the first level, the dataset was preprocessed using Python and Pandas to assure its quality for SQL querying. 
This includes the following checks and manipulations: 

- The original CSV was successfully loaded after encoding troubleshooting.
- Converting column names to snake_case for standardized and error free querying. 
- Checking for missing (or null) values. 
- Checking for redundant data entries (duplicated rows)
- Checking data types and changing data types where necessary
- Checking data for business sense (ie. sales is always positive, discounts land between 0 and 1)
- Taking the cleaned dataframe and resaved as a new CSV with UTF-8 encoding which is native to DuckDB.

In the second level, the new UTF-8 encoded CSV was loaded into DuckDB for data exploration and feature engineering. New features include:
- fulfillment_days
- profit_margin
- order_year
- order_month
- customer_lifetime_sales


In the third level, business analytics and insights are formed. The data manipulation is used to answer questions that are necessary to help stakeholders make decisions. 



Raw Data  
↓  
Preprocessing  
↓  
Feature Creation  
↓  
Analysis  
↓  
Conclusions



## Level 1 - Dataset preprocessing using Python and Pandas

#### Load Dataset

Import the dataset into Pandas and perform an initial inspection of its structure.

In [1]:
import pandas as pd

df = pd.read_csv("/Users/danigeiger/projects/e_commerce_duckdb_project/data/superstore.csv", encoding="cp1252")

df.head()


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,11/8/2016,11/11/2016,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,6/12/2016,6/16/2016,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,10/11/2015,10/18/2015,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


#### Standardize column names

Standardize column names using snake_case formatting to improve readability and maintain consistency throughout the project.

In [2]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

####  Check That Data Types Are Properly Attributed

Column data types were reviewed to confirm that numerical, categorical, and date fields were correctly interpreted by Pandas.

In [3]:
df.dtypes

row_id             int64
order_id             str
order_date           str
ship_date            str
ship_mode            str
customer_id          str
customer_name        str
segment              str
country              str
city                 str
state                str
postal_code        int64
region               str
product_id           str
category             str
sub_category         str
product_name         str
sales            float64
quantity           int64
discount         float64
profit           float64
dtype: object

#### Convert Postal Codes to Strings

Postal codes were converted to string format to preserve leading zeros and prevent geographic identifiers from being treated as numerical values.

In [4]:
df["postal_code"] = df["postal_code"].astype(str)
df["postal_code"].dtype

<StringDtype(storage='python', na_value=nan)>

#### Convert Dates to Datetime Data Types

Date columns were converted to datetime format to support time-based calculations and temporal analysis.

In [5]:
df["order_date"] = pd.to_datetime(df["order_date"])
df["ship_date"] = pd.to_datetime(df["ship_date"])
df[["order_date", "ship_date"]].dtypes


order_date    datetime64[us]
ship_date     datetime64[us]
dtype: object

#### Check for Missing Values
 Missing values were assessed across all columns to identify incomplete records that could impact downstream analysis.

In [6]:
df.isnull().sum()

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

#### Check for Duplicated Rows

Duplicate records were evaluated to ensure that transactions were not counted more than once.

In [7]:
print(df.duplicated().sum())

0


### Business Logic Validation

#### Validate Customer Identifier Integrity

Customer identifiers were validated to ensure a one-to-one relationship between customer_id and customer_name. This check confirms that each customer ID maps to exactly one customer name and helps identify potential data quality issues such as duplicate identifiers, inconsistent naming conventions, or customer records associated with multiple names.

In [8]:
customer_mapping = df.groupby('customer_id')['customer_name'].nunique()

(customer_mapping == 1).all()

np.True_

#### Check That Order Dates Always Preceded Ship Dates

Business logic validation was performed to ensure that orders were not shipped before they were placed.

In [9]:
(df['ship_date'] < df['order_date']).sum()

np.int64(0)

#### Check That Sales and Quantity Are Positive

Sales and quantity fields were reviewed to verify that transaction records contained valid positive values.

In [10]:
print(f'There are {(df["sales"] < 0).sum()} negative sales entries and {(df["quantity"] < 0).sum()} negative quantity entries.')

There are 0 negative sales entries and 0 negative quantity entries.


#### Check That All Discounts Fall Between 0 and 1

Discount values were validated to ensure they fell within the expected range of 0% to 100%.

In [11]:
print(df['discount'].describe()[['min', 'max']])


min    0.0
max    0.8
Name: discount, dtype: float64


### Categorical Validation

#### Product Hierarchy Validation

Unique values were reviewed for Category and Sub-Category to identify spelling errors, inconsistent capitalization, duplicate labels, or unexpected product groupings.

In [12]:
print("Category:")
print(sorted(df['category'].unique()))

print("\nSub-Category:")
print(sorted(df['sub_category'].unique()))



Category:
['Furniture', 'Office Supplies', 'Technology']

Sub-Category:
['Accessories', 'Appliances', 'Art', 'Binders', 'Bookcases', 'Chairs', 'Copiers', 'Envelopes', 'Fasteners', 'Furnishings', 'Labels', 'Machines', 'Paper', 'Phones', 'Storage', 'Supplies', 'Tables']


#### Geographic Validation

Country, Region, and State values were inspected to verify geographic consistency and identify any spelling, capitalization, or formatting issues.

In [13]:
print("Country:")
print(sorted(df['country'].unique()))

print("\nRegion:")
print(sorted(df['region'].unique()))

print("\nState:")
print(sorted(df['state'].unique()))

Country:
['United States']

Region:
['Central', 'East', 'South', 'West']

State:
['Alabama', 'Arizona', 'Arkansas', 'California', 'Colorado', 'Connecticut', 'Delaware', 'District of Columbia', 'Florida', 'Georgia', 'Idaho', 'Illinois', 'Indiana', 'Iowa', 'Kansas', 'Kentucky', 'Louisiana', 'Maine', 'Maryland', 'Massachusetts', 'Michigan', 'Minnesota', 'Mississippi', 'Missouri', 'Montana', 'Nebraska', 'Nevada', 'New Hampshire', 'New Jersey', 'New Mexico', 'New York', 'North Carolina', 'North Dakota', 'Ohio', 'Oklahoma', 'Oregon', 'Pennsylvania', 'Rhode Island', 'South Carolina', 'South Dakota', 'Tennessee', 'Texas', 'Utah', 'Vermont', 'Virginia', 'Washington', 'West Virginia', 'Wisconsin', 'Wyoming']


#### Customer and Fulfillment Validation

Customer segment and shipping mode categories were reviewed to ensure consistency across customer classifications and order fulfillment methods.

In [14]:
print("Ship Mode:")
print(sorted(df['ship_mode'].unique()))

print("\nSegment:")
print(sorted(df['segment'].unique()))


Ship Mode:
['First Class', 'Same Day', 'Second Class', 'Standard Class']

Segment:
['Consumer', 'Corporate', 'Home Office']


### Final Validation Gate
Ensures the cleaned dataset adheres strictly to all business logic rules before exporting to DuckDB.


In [15]:
assert df.isnull().sum().sum() == 0, "Alert: Missing values detected!" #check for missing values
assert df.duplicated().sum() == 0, "Alert: Duplicate rows detected!"    #check for duplicate rows
assert df['row_id'].is_unique, "Alert: row_id contains duplicates!" #check row_id is unique

assert pd.api.types.is_datetime64_any_dtype(df['order_date']), "Alert: order_date is not datetime!" #check order_date is datetime
assert pd.api.types.is_datetime64_any_dtype(df['ship_date']), "Alert: ship_date is not datetime!" #check ship_date is datetime
assert (df['ship_date'] >= df['order_date']).all(), "Alert: Ship date occurs before order date!"    #check ship_date is not before order_date

assert (df['sales'] > 0).all(), "Alert: Found non-positive sales!" #check for non-positive sales
assert (df['quantity'] > 0).all(), "Alert: Found non-positive quantities!" #check for non-positive quantities
assert df['discount'].between(0, 1).all(), "Alert: Discounts out of 0-1 bounds!" #check discount is between 0 and 1

assert df['customer_id'].nunique() == df['customer_name'].nunique(), "Alert: Unique customer counts do not match!" #check unique customer_id matches unique customer_name


print("🎉 Setup complete. All final validation checks passed!")


🎉 Setup complete. All final validation checks passed!


#### Export Clean Dataset

Export the validated dataset as a UTF-8 encoded CSV file for use in DuckDB.

In [16]:
df.to_csv(
    "/Users/danigeiger/projects/e_commerce_duckdb_project/data/superstore_utf8.csv",
    index=False,
    encoding="utf-8"
)

# View cleaned dataset
df.head()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## Level 2 - Feature Engineering using DuckDB SQL

### Load Dataset into DuckDB

Load the cleaned dataset into DuckDB to begin feature engineering and analytical processing.

In [17]:
import duckdb

con = duckdb.connect()

# ensure program doesn't fail if view already exists
con.execute("""
CREATE OR REPLACE VIEW superstore AS        
SELECT *
FROM read_csv_auto('/Users/danigeiger/projects/e_commerce_duckdb_project/data/superstore_utf8.csv', header=True)""") # change path to relative path when uploaded to github

con.execute("""
SELECT *
FROM superstore
LIMIT 5 """).df()

,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2016-152156,2016-11-08,2016-11-11,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2016-138688,2016-06-12,2016-06-16,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2015-108966,2015-10-11,2015-10-18,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


### Feature Engineering

A DuckDB view named `superstore_features` was created to preserve the original dataset while adding analytical features. New columns include fulfillment time, profit margin, order year and month, and customer lifetime sales calculated using a window function. These engineered features support customer segmentation, profitability analysis, and operational performance reporting.

In [18]:
con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT
    *,
    date_diff('day', order_date, ship_date) AS fulfillment_days,
    profit / sales AS profit_margin,
    year(order_date) AS order_year,
    month(order_date) AS order_month,
    SUM(sales) OVER (
        PARTITION BY customer_id
    ) AS customer_lifetime_sales
FROM superstore
""")

### Quick Inspection of Newly Created Features
The newly created features were queried from the `superstore_features` view and inspected using a sample of 10 records. This validation step ensured that the feature engineering process produced the expected values before proceeding with further analysis.

In [33]:
con.execute("""
SELECT
    customer_name,
    fulfillment_days,
    profit_margin,
    order_year,
    order_month,
    customer_lifetime_sales
FROM superstore_features
LIMIT 10
""").df()

,customer_name,fulfillment_days,profit_margin,order_year,order_month,customer_lifetime_sales
0,Ann Blume,1,0.075000,2017,11,1515.862
1,Ann Blume,7,-1.650000,2014,11,1515.862
2,Ann Blume,3,0.062500,2015,11,1515.862
3,Ann Blume,1,-0.150000,2017,11,1515.862
4,Ann Blume,1,-0.666667,2017,11,1515.862
5,Ann Blume,1,-0.666667,2017,11,1515.862
6,Ann Blume,4,0.480000,2015,2,1515.862
7,Ann Blume,1,-0.162500,2017,11,1515.862
8,Aaron Hawkins,3,0.375000,2014,10,1744.700
9,Aaron Hawkins,2,0.337500,2014,4,1744.700


### Fulfillment Days Analysis

The distribution of the engineered `fulfillment_days` feature was examined to verify that the calculated values were reasonable. Most orders were fulfilled within **4–5 days**, with relatively few orders requiring **0–1 days** or the maximum of **7 days**, indicating a realistic distribution suitable for downstream operational analysis.

In [43]:
con.execute('''
SELECT
    fulfillment_days,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY fulfillment_days
ORDER BY fulfillment_days;
''').df()            

,fulfillment_days,orders
0,0,519
1,1,369
2,2,1334
3,3,1005
4,4,2774
5,5,2169
6,6,1203
7,7,621


### Profit Margin Analysis

Analysis of the engineered `profit_margin` feature showed an average profit margin of **12.03%**, indicating that the company earned approximately 12 cents of profit for every dollar of sales. Profit margins ranged from **-275%** to **50%**, highlighting that while some transactions were highly profitable, others resulted in substantial losses.

In [35]:
con.execute('''
SELECT
    ROUND(AVG(profit_margin),4) AS avg_profit_margin,
    MIN(profit_margin) AS min_margin,
    MEDIAN(profit_margin) AS median_margin,
    MAX(profit_margin) AS max_margin
FROM superstore_features;
''').df()

,avg_profit_margin,min_margin,median_margin,max_margin
0,0.1203,-2.75,0.27,0.5


### Lowest Profit Margin Products

The products with the lowest profit margins were identified to better understand the transactions contributing to overall losses. Several products exhibited profit margins below **-270%**, indicating that the losses incurred on these sales substantially exceeded the revenue generated, making them potential candidates for further pricing or discount analysis.

In [22]:
con.execute(''' 
SELECT
    product_name,
    sales,
    profit,
    profit_margin
FROM superstore_features
ORDER BY profit_margin
LIMIT 10
''').df()

,product_name,sales,profit,profit_margin
0,Kensington 6 Outlet SmartSocket Surge Protector,24.588,-67.6170,-2.75
1,Hoover Portapower Portable Vacuum,2.688,-7.3920,-2.75
2,Hoover Shoulder Vac Commercial Portable Vacuum,143.128,-393.6020,-2.75
3,Eureka Disposable Bags for Sanitaire Vibra Gro...,1.624,-4.4660,-2.75
4,Hoover Commercial Lightweight Upright Vacuum w...,93.032,-251.1864,-2.70
5,Belkin 7-Outlet SurgeMaster Home Series,5.588,-15.0876,-2.70
6,Belkin 6 Outlet Metallic Surge Strip,4.356,-11.7612,-2.70
7,Acco Smartsocket Color-Coded Six-Outlet AC Ada...,26.406,-71.2962,-2.70
8,Fellowes 8 Outlet Superior Workstation Surge P...,33.620,-90.7740,-2.70
9,Tripp Lite Isotel 8 Ultra 8 Outlet Metal Surge,70.970,-191.6190,-2.70


### Customer Lifetime Sales Analysis

The `customer_lifetime_sales` feature was used to identify the highest-value customers in the dataset. Sean Miller generated over **$25,000** in lifetime sales, while several other customers exceeded **$12,000** in total purchases, indicating that a relatively small group of customers contributed substantially to overall revenue.

In [23]:
con.execute('''
SELECT DISTINCT
    customer_name,
    customer_lifetime_sales
FROM superstore_features
ORDER BY customer_lifetime_sales DESC
LIMIT 10;
''').df()

,customer_name,customer_lifetime_sales
0,Sean Miller,25043.050
1,Tamara Chand,19052.218
2,Raymond Buch,15117.339
3,Tom Ashbrook,14595.620
4,Adrian Barton,14473.571
5,Ken Lonsdale,14175.229
6,Sanjit Chand,14142.334
7,Hunter Lopez,12873.298
8,Sanjit Engle,12209.438
9,Christopher Conant,12129.072


In [24]:
con.execute('''
SELECT
    quantile_cont(customer_lifetime_sales, 0.25) AS q1,
    quantile_cont(customer_lifetime_sales, 0.50) AS median,
    quantile_cont(customer_lifetime_sales, 0.75) AS q3,
    quantile_cont(customer_lifetime_sales, 0.9) AS top_10,
FROM (
    SELECT DISTINCT
        customer_id,
        customer_lifetime_sales
    FROM superstore_features
);
''').df()

,q1,median,q3,top_10
0,1146.05,2256.394,3785.276,6038.48


In [25]:
con.execute("""
CREATE OR REPLACE VIEW superstore_features AS
SELECT
    *,
    CASE
        WHEN customer_lifetime_sales >= 6038.48 THEN 'Big Fish'
        WHEN customer_lifetime_sales >= 3785.276 THEN 'Premium'
        WHEN customer_lifetime_sales >= 2256.394 THEN 'High Value'
        ELSE 'Standard'
    END AS customer_tier
FROM (
    SELECT
        *,
        date_diff('day', order_date, ship_date) AS fulfillment_days,
        profit / sales AS profit_margin,
        year(order_date) AS order_year,
        month(order_date) AS order_month,
        SUM(sales) OVER (
            PARTITION BY customer_id
        ) AS customer_lifetime_sales
    FROM superstore
);
""")

### Customer Segmentation

Customers were segmented according to their lifetime sales using percentile-based thresholds. Customers in the top 10% of lifetime spending were classified as "Big Fish," while the remaining customers were categorized as Premium, High Value, or Standard based on the 75th and 50th percentile cutoffs. This feature provides a business-oriented method for identifying and analyzing high-value customers.

In [26]:
con.execute('''
SELECT DISTINCT
        customer_name,
        customer_lifetime_sales,
        customer_tier
FROM superstore_features
ORDER BY customer_lifetime_sales DESC
LIMIT 10;
''').df()

,customer_name,customer_lifetime_sales,customer_tier
0,Sean Miller,25043.050,Big Fish
1,Tamara Chand,19052.218,Big Fish
2,Raymond Buch,15117.339,Big Fish
3,Tom Ashbrook,14595.620,Big Fish
4,Adrian Barton,14473.571,Big Fish
5,Ken Lonsdale,14175.229,Big Fish
6,Sanjit Chand,14142.334,Big Fish
7,Hunter Lopez,12873.298,Big Fish
8,Sanjit Engle,12209.438,Big Fish
9,Christopher Conant,12129.072,Big Fish


In [27]:
con.execute('''
SELECT
    order_year,
    COUNT(*) AS orders
FROM superstore_features
GROUP BY order_year
ORDER BY order_year;
''').df()

,order_year,orders
0,2014,1993
1,2015,2102
2,2016,2587
3,2017,3312


## Level 3 - Business insights using DuckDB SQL

### Do Product Prices Differ Across Customer Segments?

Average unit prices and discounts were compared across customer segments to determine whether pricing patterns differed between Consumer, Corporate, and Home Office customers.

In [28]:
con.execute('''
SELECT 
    product_id,
    segment,
    ROUND(AVG(sales / quantity), 2) AS avg_unit_price,
    ROUND(AVG(discount), 3) AS avg_discount
FROM superstore
GROUP BY product_id, segment
ORDER BY product_id
Limit 40''').df()           


,product_id,segment,avg_unit_price,avg_discount
0,FUR-BO-10000112,Corporate,91.69,0.300
1,FUR-BO-10000330,Consumer,111.91,0.075
2,FUR-BO-10000330,Home Office,102.83,0.150
3,FUR-BO-10000362,Home Office,145.33,0.150
4,FUR-BO-10000362,Consumer,136.78,0.200
5,FUR-BO-10000362,Corporate,158.16,0.075
6,FUR-BO-10000468,Consumer,37.89,0.220
7,FUR-BO-10000468,Corporate,48.58,0.000
8,FUR-BO-10000711,Home Office,70.98,0.000
9,FUR-BO-10000711,Consumer,70.98,0.000


In [29]:
con.execute ("""
SELECT 
    segment,
    COUNT(DISTINCT customer_name) AS customer_count,
    ROUND(SUM(sales)) AS total_sales
FROM superstore
GROUP BY segment
""").df()




,segment,customer_count,total_sales
0,Home Office,148,429653.0
1,Consumer,409,1161401.0
2,Corporate,236,706146.0


In [30]:
con.execute('''
SELECT
    product_id,
    segment,
    ROUND(AVG(sales / quantity), 2) AS avg_unit_price,
    ROUND(AVG(discount), 3) AS avg_discount
FROM superstore
GROUP BY product_id, segment
ORDER BY product_id
LIMIT 40;
''').df()

,product_id,segment,avg_unit_price,avg_discount
0,FUR-BO-10000112,Corporate,91.69,0.300
1,FUR-BO-10000330,Consumer,111.91,0.075
2,FUR-BO-10000330,Home Office,102.83,0.150
3,FUR-BO-10000362,Consumer,136.78,0.200
4,FUR-BO-10000362,Corporate,158.16,0.075
5,FUR-BO-10000362,Home Office,145.33,0.150
6,FUR-BO-10000468,Consumer,37.89,0.220
7,FUR-BO-10000468,Corporate,48.58,0.000
8,FUR-BO-10000711,Home Office,70.98,0.000
9,FUR-BO-10000711,Consumer,70.98,0.000


In [ ]:
con.execute('''
SELECT
    segment,
    ROUND(AVG(fulfillment_days),2) AS avg_fulfillment_days,
    MIN(fulfillment_days) AS min_days,
    MEDIAN(fulfillment_days) AS median_days,
    MAX(fulfillment_days) AS max_days
FROM superstore_features
GROUP BY segment
ORDER BY avg_fulfillment_days;
''').df()

In [ ]:
con.execute("""
SELECT
    ship_mode,
    ROUND(AVG(fulfillment_days), 2) AS avg_fulfillment_days,
    MIN(fulfillment_days) AS min_days,
    MAX(fulfillment_days) AS max_days
FROM superstore_features
GROUP BY ship_mode
ORDER BY avg_fulfillment_days;
""").df()